In [1]:
import uproot
import pandas as pd
import numpy as np
import ipywidgets as widgets
import plotly.graph_objects as go

file_path = "../clusters_seeds_island_79507-0.root_ntuplizer.root"
print("Loading real data for interactive labeling...")
with uproot.open(file_path) as f:
    hit_tree = f["ntp_hit"]
    df_real = hit_tree.arrays(["event", "hitID", "layer", "phi", "tbin", "zelem", "x", "y", "z", "adc"], library="pd")

    print(f"Initial hits loaded: {len(df_real)}")

    # 1. Isolate the TPC layers
    df_real = df_real[(df_real['layer'] >= 7) & (df_real['layer'] <= 54)]

    # 2. Keep only signal/truth-associated hits (no background noise)
    df_real = df_real[df_real['hitID'] > 0]

    # 3. Ensure no zero-energy ghost hits
    df_real = df_real[df_real['adc'] > 0]

    # 4. Filter NaNs
    df_real = df_real.dropna(subset=['x', 'y', 'z', 'phi', 'tbin'])

    # 5. Enforce zelem bounds for TPC
    df_real = df_real[df_real['zelem'].isin([0, 1])]

    print(f"Valid TPC hits remaining after cuts: {len(df_real)}")


df_real['label'] = 'unknown'
df_real['color'] = 'gray'

# Pre-compute transformed coordinates
df_real['trans_x'] = df_real['layer'] * np.cos(df_real['phi'])
df_real['trans_y'] = df_real['layer'] * np.sin(df_real['phi'])

print(f"Successfully loaded {len(df_real)} hits. Ready for labeling!")

class LabelingUI:
    def __init__(self, mode="xyz"):
        self.mode = mode
        
        self.fig3d = go.FigureWidget()
        scatter3d = go.Scatter3d(mode='markers', marker=dict(size=3 if mode!='layer_phi_tbin' else 2), hoverinfo='text')
        self.fig3d.add_trace(scatter3d)
        
        self.fig2d = go.FigureWidget()
        scatter2d = go.Scatter(mode='markers', marker=dict(size=4 if mode!='layer_phi_tbin' else 5), hoverinfo='text')
        self.fig2d.add_trace(scatter2d)
        
        if mode == "xyz":
            self.fig3d.update_layout(title="3D View (Selected Event & Layer)", margin=dict(l=0, r=0, b=0, t=40), scene=dict(aspectmode='data'))
            self.fig2d.update_layout(title="2D Slice: X vs Y", dragmode='lasso', margin=dict(l=0, r=0, b=0, t=40), xaxis_title="X", yaxis_title="Y")
        elif mode == "transformed":
            self.fig3d.update_layout(title="3D View (Transformed: X/Y=r*phi, Z=tbin)", margin=dict(l=0, r=0, b=0, t=40), scene=dict(aspectmode='data'))
            self.fig2d.update_layout(title="2D Slice: X vs Y", dragmode='lasso', margin=dict(l=0, r=0, b=0, t=40), xaxis_title="X", yaxis_title="Y")
        elif mode == "layer_phi_tbin":
            self.fig3d.update_layout(title="3D View (X=Layer, Y=Phi, Z=Tbin)", margin=dict(l=0, r=0, b=0, t=40), 
                                scene=dict(xaxis_title='Layer', yaxis_title='Phi', zaxis_title='Tbin', aspectmode='data'))
            self.fig2d.update_layout(title="2D Slice: Phi vs Tbin", dragmode='lasso', margin=dict(l=0, r=0, b=0, t=40), xaxis_title="Phi", yaxis_title="Tbin")

        self.trace3d = self.fig3d.data[0]
        self.trace2d = self.fig2d.data[0]
        
        available_events = sorted(df_real['event'].unique().astype(int))
        self.event_dropdown = widgets.Dropdown(options=available_events, value=available_events[0], description='Event:')
        
        available_layers = sorted(df_real['layer'].unique().astype(int))
        self.layer_dropdown = widgets.Dropdown(options=available_layers, value=available_layers[0], description='Layer:')

        self.event_dropdown.observe(self.update_plots, 'value')
        self.layer_dropdown.observe(self.update_plots, 'value')

        self.trace2d.on_selection(self.on_selection)
        self.trace2d.on_click(self.on_click)
        self.trace3d.on_click(self.on_click)

        self.update_plots()

        self.ui = widgets.VBox([
            widgets.HBox([self.event_dropdown, self.layer_dropdown]),
            widgets.HBox([self.fig2d, self.fig3d])
        ])

    def update_plots(self, *args):
        current_event = self.event_dropdown.value
        current_layer = self.layer_dropdown.value
        df_slice = df_real[(df_real['event'] == current_event) & (df_real['layer'] == current_layer)]
        
        if self.mode == "xyz":
            x3, y3, z3 = df_slice['x'], df_slice['y'], df_slice['z']
            x2, y2 = df_slice['x'], df_slice['y']
            hover = [f"Index: {i}<br>ADC: {adc:.1f}<br>Label: {lbl}" for i, adc, lbl in zip(df_slice.index, df_slice['adc'], df_slice['label'])]
        elif self.mode == "transformed":
            x3, y3, z3 = df_slice['trans_x'], df_slice['trans_y'], df_slice['tbin']
            x2, y2 = df_slice['trans_x'], df_slice['trans_y']
            hover = [f"Index: {i}<br>ADC: {adc:.1f}<br>Label: {lbl}" for i, adc, lbl in zip(df_slice.index, df_slice['adc'], df_slice['label'])]
        elif self.mode == "layer_phi_tbin":
            x3, y3, z3 = df_slice['layer'], df_slice['phi'], df_slice['tbin']
            x2, y2 = df_slice['phi'], df_slice['tbin']
            hover = [f"Index: {i}<br>ADC: {row.adc:.1f}<br>x: {row.x:.2f}<br>y: {row.y:.2f}<br>z: {row.z:.2f}<br>Label: {row.label}" for i, row in df_slice.iterrows()]
            
        with self.fig3d.batch_update():
            self.trace3d.x = x3
            self.trace3d.y = y3
            self.trace3d.z = z3
            self.trace3d.marker.color = df_slice['color'].tolist()
            self.trace3d.customdata = df_slice.index
            self.trace3d.text = hover
            
        with self.fig2d.batch_update():
            self.trace2d.x = x2
            self.trace2d.y = y2
            self.trace2d.marker.color = df_slice['color'].tolist()
            self.trace2d.customdata = df_slice.index
            self.trace2d.text = hover

    def update_labels_and_colors(self, points, new_label, new_color):
        if not points.point_inds:
            return
        for idx in points.point_inds:
            df_row_idx = self.trace2d.customdata[idx]
            df_real.loc[df_row_idx, 'label'] = new_label
            df_real.loc[df_row_idx, 'color'] = new_color
        self.update_plots()

    def on_selection(self, trace, points, selector):
        if not points.point_inds:
            return
        for idx in points.point_inds:
            df_row_idx = trace.customdata[idx]
            current_label = df_real.loc[df_row_idx, 'label']
            if current_label == 'unknown':
                df_real.loc[df_row_idx, 'label'] = 'signal'
                df_real.loc[df_row_idx, 'color'] = 'red'
            elif current_label == 'signal':
                df_real.loc[df_row_idx, 'label'] = 'noise'
                df_real.loc[df_row_idx, 'color'] = 'blue'
            else:
                df_real.loc[df_row_idx, 'label'] = 'unknown'
                df_real.loc[df_row_idx, 'color'] = 'gray'
        self.update_plots()

    def on_click(self, trace, points, selector):
        if not points.point_inds:
            return
        idx = points.point_inds[0]
        df_row_idx = trace.customdata[idx]
        current_label = df_real.loc[df_row_idx, 'label']
        if current_label == 'unknown':
            self.update_labels_and_colors(points, 'signal', 'red')
        elif current_label == 'signal':
            self.update_labels_and_colors(points, 'noise', 'blue')
        else:
            self.update_labels_and_colors(points, 'unknown', 'gray')

# Apply the 5 bulletproof data cleaning cuts
print(f"Initial hits loaded: {len(df_real)}")

# 1. Isolate the TPC layers
df_real = df_real[(df_real['layer'] >= 7) & (df_real['layer'] <= 54)]

# 2. Keep only signal/truth-associated hits (no background noise)
df_real = df_real[df_real['hitID'] > 0]

# 3. Ensure no zero-energy ghost hits
df_real = df_real[df_real['adc'] > 0]

# 4. Filter NaNs
df_real = df_real.dropna(subset=['x', 'y', 'z', 'phi', 'tbin'])

# 5. Enforce zelem bounds for TPC
df_real = df_real[df_real['zelem'].isin([0, 1])]

print(f"Valid TPC hits remaining after cuts: {len(df_real)}")


df_real['label'] = 'unknown'
df_real['color'] = 'gray'

# Pre-compute transformed coordinates
df_real['trans_x'] = df_real['layer'] * np.cos(df_real['phi'])
df_real['trans_y'] = df_real['layer'] * np.sin(df_real['phi'])

print(f"Successfully loaded {len(df_real)} hits. Ready for labeling!")

class LabelingUI:
    def __init__(self, mode="xyz"):
        self.mode = mode
        
        self.fig3d = go.FigureWidget()
        scatter3d = go.Scatter3d(mode='markers', marker=dict(size=3 if mode!='layer_phi_tbin' else 2), hoverinfo='text')
        self.fig3d.add_trace(scatter3d)
        
        self.fig2d = go.FigureWidget()
        scatter2d = go.Scatter(mode='markers', marker=dict(size=4 if mode!='layer_phi_tbin' else 5), hoverinfo='text')
        self.fig2d.add_trace(scatter2d)
        
        if mode == "xyz":
            self.fig3d.update_layout(title="3D View (Selected Event & Layer)", margin=dict(l=0, r=0, b=0, t=40), scene=dict(aspectmode='data'))
            self.fig2d.update_layout(title="2D Slice: X vs Y", dragmode='lasso', margin=dict(l=0, r=0, b=0, t=40), xaxis_title="X", yaxis_title="Y")
        elif mode == "transformed":
            self.fig3d.update_layout(title="3D View (Transformed: X/Y=r*phi, Z=tbin)", margin=dict(l=0, r=0, b=0, t=40), scene=dict(aspectmode='data'))
            self.fig2d.update_layout(title="2D Slice: X vs Y", dragmode='lasso', margin=dict(l=0, r=0, b=0, t=40), xaxis_title="X", yaxis_title="Y")
        elif mode == "layer_phi_tbin":
            self.fig3d.update_layout(title="3D View (X=Layer, Y=Phi, Z=Tbin)", margin=dict(l=0, r=0, b=0, t=40), 
                                scene=dict(xaxis_title='Layer', yaxis_title='Phi', zaxis_title='Tbin', aspectmode='data'))
            self.fig2d.update_layout(title="2D Slice: Phi vs Tbin", dragmode='lasso', margin=dict(l=0, r=0, b=0, t=40), xaxis_title="Phi", yaxis_title="Tbin")

        self.trace3d = self.fig3d.data[0]
        self.trace2d = self.fig2d.data[0]
        
        available_events = sorted(df_real['event'].unique().astype(int))
        self.event_dropdown = widgets.Dropdown(options=available_events, value=available_events[0], description='Event:')
        
        available_layers = sorted(df_real['layer'].unique().astype(int))
        self.layer_dropdown = widgets.Dropdown(options=available_layers, value=available_layers[0], description='Layer:')

        self.event_dropdown.observe(self.update_plots, 'value')
        self.layer_dropdown.observe(self.update_plots, 'value')

        self.trace2d.on_selection(self.on_selection)
        self.trace2d.on_click(self.on_click)
        self.trace3d.on_click(self.on_click)

        self.update_plots()

        self.ui = widgets.VBox([
            widgets.HBox([self.event_dropdown, self.layer_dropdown]),
            widgets.HBox([self.fig2d, self.fig3d])
        ])

    def update_plots(self, *args):
        current_event = self.event_dropdown.value
        current_layer = self.layer_dropdown.value
        df_slice = df_real[(df_real['event'] == current_event) & (df_real['layer'] == current_layer)]
        
        if self.mode == "xyz":
            x3, y3, z3 = df_slice['x'], df_slice['y'], df_slice['z']
            x2, y2 = df_slice['x'], df_slice['y']
            hover = [f"Index: {i}<br>ADC: {adc:.1f}<br>Label: {lbl}" for i, adc, lbl in zip(df_slice.index, df_slice['adc'], df_slice['label'])]
        elif self.mode == "transformed":
            x3, y3, z3 = df_slice['trans_x'], df_slice['trans_y'], df_slice['tbin']
            x2, y2 = df_slice['trans_x'], df_slice['trans_y']
            hover = [f"Index: {i}<br>ADC: {adc:.1f}<br>Label: {lbl}" for i, adc, lbl in zip(df_slice.index, df_slice['adc'], df_slice['label'])]
        elif self.mode == "layer_phi_tbin":
            x3, y3, z3 = df_slice['layer'], df_slice['phi'], df_slice['tbin']
            x2, y2 = df_slice['phi'], df_slice['tbin']
            hover = [f"Index: {i}<br>ADC: {row.adc:.1f}<br>x: {row.x:.2f}<br>y: {row.y:.2f}<br>z: {row.z:.2f}<br>Label: {row.label}" for i, row in df_slice.iterrows()]
            
        with self.fig3d.batch_update():
            self.trace3d.x = x3
            self.trace3d.y = y3
            self.trace3d.z = z3
            self.trace3d.marker.color = df_slice['color'].tolist()
            self.trace3d.customdata = df_slice.index
            self.trace3d.text = hover
            
        with self.fig2d.batch_update():
            self.trace2d.x = x2
            self.trace2d.y = y2
            self.trace2d.marker.color = df_slice['color'].tolist()
            self.trace2d.customdata = df_slice.index
            self.trace2d.text = hover

    def update_labels_and_colors(self, points, new_label, new_color):
        if not points.point_inds:
            return
        for idx in points.point_inds:
            df_row_idx = self.trace2d.customdata[idx]
            df_real.loc[df_row_idx, 'label'] = new_label
            df_real.loc[df_row_idx, 'color'] = new_color
        self.update_plots()

    def on_selection(self, trace, points, selector):
        if not points.point_inds:
            return
        for idx in points.point_inds:
            df_row_idx = trace.customdata[idx]
            current_label = df_real.loc[df_row_idx, 'label']
            if current_label == 'unknown':
                df_real.loc[df_row_idx, 'label'] = 'signal'
                df_real.loc[df_row_idx, 'color'] = 'red'
            elif current_label == 'signal':
                df_real.loc[df_row_idx, 'label'] = 'noise'
                df_real.loc[df_row_idx, 'color'] = 'blue'
            else:
                df_real.loc[df_row_idx, 'label'] = 'unknown'
                df_real.loc[df_row_idx, 'color'] = 'gray'
        self.update_plots()

    def on_click(self, trace, points, selector):
        if not points.point_inds:
            return
        idx = points.point_inds[0]
        df_row_idx = trace.customdata[idx]
        current_label = df_real.loc[df_row_idx, 'label']
        if current_label == 'unknown':
            self.update_labels_and_colors(points, 'signal', 'red')
        elif current_label == 'signal':
            self.update_labels_and_colors(points, 'noise', 'blue')
        else:
            self.update_labels_and_colors(points, 'unknown', 'gray')


Loading real data for interactive labeling...
Initial hits loaded: 27532828
Valid TPC hits remaining after cuts: 25671334
Successfully loaded 25671334 hits. Ready for labeling!
Initial hits loaded: 25671334
Valid TPC hits remaining after cuts: 25671334
Successfully loaded 25671334 hits. Ready for labeling!


In [2]:
ui_xyz = LabelingUI("xyz")
display(ui_xyz.ui)

In [3]:
ui_trans = LabelingUI("transformed")
display(ui_trans.ui)

In [4]:
ui_lpt = LabelingUI("layer_phi_tbin")
display(ui_lpt.ui)